# Inheritance Patterns

When AI code has `class Child(Parent)`, what does that mean? Let's understand inheritance.

## Basic Inheritance

A child class gets everything from the parent class and can add or override.

In [ ]:
class Animal:
    """Base class for all animals."""
    
    def __init__(self, name):
        self.name = name
    
    def speak(self):
        return "Some sound"
    
    def describe(self):
        return f"{self.name} is an animal"


class Dog(Animal):  # Dog inherits from Animal
    """A dog is a specific type of animal."""
    
    def speak(self):  # Override parent method
        return "Woof!"
    
    def fetch(self):  # Add new method
        return f"{self.name} fetches the ball"


class Cat(Animal):
    def speak(self):
        return "Meow!"


# Usage
dog = Dog("Buddy")
cat = Cat("Whiskers")

print(dog.speak())      # "Woof!" - overridden
print(dog.describe())   # From Animal - inherited
print(dog.fetch())      # Dog-specific
print(cat.speak())      # "Meow!"

## Understanding `super()`

**`super()` calls the parent class's method.** This is essential for proper initialization.

In [ ]:
class Vehicle:
    def __init__(self, brand, model):
        self.brand = brand
        self.model = model
        print(f"Vehicle.__init__ called: {brand} {model}")


class Car(Vehicle):
    def __init__(self, brand, model, num_doors):
        # Call parent's __init__ to set brand and model
        super().__init__(brand, model)
        # Then set Car-specific attributes
        self.num_doors = num_doors
        print(f"Car.__init__ called: {num_doors} doors")


class ElectricCar(Car):
    def __init__(self, brand, model, num_doors, battery_kwh):
        # Call Car's __init__ (which calls Vehicle's)
        super().__init__(brand, model, num_doors)
        self.battery_kwh = battery_kwh
        print(f"ElectricCar.__init__ called: {battery_kwh} kWh")


# Watch the chain of __init__ calls
tesla = ElectricCar("Tesla", "Model 3", 4, 75)
print(f"\nResult: {tesla.brand} {tesla.model}, {tesla.num_doors} doors, {tesla.battery_kwh} kWh")

## Why `super()` Matters

**Without `super()`**, parent initialization is skipped:

In [ ]:
class Parent:
    def __init__(self):
        self.parent_attr = "I'm from Parent"

class BadChild(Parent):
    def __init__(self):
        # Forgot super().__init__()!
        self.child_attr = "I'm from Child"

class GoodChild(Parent):
    def __init__(self):
        super().__init__()  # Don't forget this!
        self.child_attr = "I'm from Child"

bad = BadChild()
good = GoodChild()

print(f"GoodChild has parent_attr: {hasattr(good, 'parent_attr')}")  # True
print(f"BadChild has parent_attr: {hasattr(bad, 'parent_attr')}")    # False!

## Abstract Base Classes (ABCs)

ABCs define a "contract" that child classes must follow:

In [ ]:
from abc import ABC, abstractmethod

class DataLoader(ABC):
    """Abstract base class - cannot be instantiated directly."""
    
    @abstractmethod
    def load(self) -> list:
        """Subclasses MUST implement this method."""
        pass
    
    def preview(self, n=5):
        """Concrete method - inherited by all subclasses."""
        data = self.load()
        return data[:n]


class JSONLoader(DataLoader):
    def __init__(self, data):
        self.data = data
    
    def load(self) -> list:  # Must implement this!
        return self.data


class CSVLoader(DataLoader):
    def __init__(self, rows):
        self.rows = rows
    
    def load(self) -> list:
        return self.rows


# Usage
json_loader = JSONLoader([1, 2, 3, 4, 5, 6, 7])
print(json_loader.preview(3))  # Uses inherited preview()

# This would fail:
# loader = DataLoader()  # TypeError: Can't instantiate abstract class

## Multiple Inheritance and Mixins

Python allows inheriting from multiple classes:

In [ ]:
# Mixins add functionality without being full classes

class JSONMixin:
    """Mixin that adds JSON serialization."""
    def to_json(self):
        import json
        return json.dumps(self.__dict__)


class PrintableMixin:
    """Mixin that adds pretty printing."""
    def pretty_print(self):
        attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{self.__class__.__name__}({attrs})"


class User(JSONMixin, PrintableMixin):
    """User class with mixin capabilities."""
    def __init__(self, name, email):
        self.name = name
        self.email = email


user = User("Alice", "alice@example.com")
print(user.to_json())       # From JSONMixin
print(user.pretty_print())  # From PrintableMixin

## Method Resolution Order (MRO)

With multiple inheritance, Python needs to know which method to call:

In [ ]:
class A:
    def method(self):
        return "A"

class B(A):
    def method(self):
        return "B"

class C(A):
    def method(self):
        return "C"

class D(B, C):  # Inherits from both B and C
    pass

d = D()
print(f"d.method() returns: {d.method()}")  # Which one?

# See the Method Resolution Order
print(f"MRO: {[cls.__name__ for cls in D.__mro__]}")
# D -> B -> C -> A -> object

## Common AI Code Patterns

### Pattern: Base class with abstract methods
```python
class BaseHandler(ABC):
    @abstractmethod
    def handle(self, request):
        pass

class JSONHandler(BaseHandler):
    def handle(self, request):
        return json.loads(request)
```

### Pattern: Mixin for common functionality
```python
class TimestampMixin:
    created_at: datetime = field(default_factory=datetime.now)

class User(TimestampMixin, BaseModel):
    name: str
```

## Summary

| Concept | Syntax | Purpose |
|---------|--------|--------|
| Inheritance | `class Child(Parent)` | Extend/specialize a class |
| `super()` | `super().__init__()` | Call parent method |
| Override | Same method name in child | Replace parent behavior |
| ABC | `class X(ABC)` + `@abstractmethod` | Define interface/contract |
| Multiple inheritance | `class X(A, B)` | Combine classes |
| Mixin | Small class for one feature | Add capabilities |
| MRO | `Class.__mro__` | Method lookup order |

## Next Up

Dataclasses - the modern shortcut for creating classes.

Continue to: [Dataclasses](03-dataclasses.ipynb)